# Exercice 1 — Détection de fraude bancaire

**Objectif** : développer un système de détection automatique des transactions frauduleuses à partir du dataset `detection_fraude.csv`.

**Plan :**
1. Analyse exploratoire (EDA)
2. Prétraitement & feature engineering
3. Gestion du déséquilibre des classes
4. Modélisation (Logistic Regression, Random Forest, XGBoost, LightGBM, NN)
5. Évaluation & comparaison
6. Interprétabilité (SHAP)
7. Analyse des erreurs

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from src.preprocessing import load_fraud_data, engineer_fraud_features, encode_transaction_type
from src.models import FRAUD_MODELS, train_and_evaluate, cross_validate_models
from src.utils import plot_class_distribution, plot_confusion_matrix, plot_roc_curve, evaluate_classifier

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

DATA_PATH = '../data/raw/detection_fraude.csv'
RANDOM_STATE = 42

## 1. Analyse exploratoire des données (EDA)

In [ ]:
df = load_fraud_data(DATA_PATH)
print(f"Dimensions : {df.shape}")
df.head()

In [ ]:
print("\n--- Informations générales ---")
df.info()
print("\n--- Valeurs manquantes ---")
print(df.isnull().sum())
print("\n--- Statistiques descriptives ---")
df.describe()

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_class_distribution(df['isFraud'], title='Distribution isFraud', ax=axes[0])

# Répartition par type de transaction
df.groupby(['type', 'isFraud']).size().unstack(fill_value=0).plot(
    kind='bar', ax=axes[1], color=['steelblue', 'tomato']
)
axes[1].set_title('Fraudes par type de transaction')
axes[1].set_xlabel('Type')
axes[1].legend(['Normal', 'Fraude'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/ex1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution des montants
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['isFraud'] == 0]['amount'].clip(upper=df['amount'].quantile(0.99)).hist(
    bins=50, ax=axes[0], color='steelblue', alpha=0.7, label='Normal'
)
df[df['isFraud'] == 1]['amount'].hist(
    bins=50, ax=axes[0], color='tomato', alpha=0.7, label='Fraude'
)
axes[0].set_title('Distribution des montants')
axes[0].legend()
axes[0].set_xlabel('Montant (clippé au 99e percentile)')

# Montant moyen par type (fraude vs normal)
df.groupby(['type', 'isFraud'])['amount'].mean().unstack().plot(
    kind='bar', ax=axes[1], color=['steelblue', 'tomato']
)
axes[1].set_title('Montant moyen par type et classe')
axes[1].legend(['Normal', 'Fraude'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/ex1_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comportements suspects : soldes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'tomato', 'Fraude')]:
    subset = df[df['isFraud'] == label]
    axes[0].scatter(subset['oldbalanceOrg'].clip(upper=1e6),
                    subset['newbalanceOrig'].clip(upper=1e6),
                    alpha=0.1, s=2, color=color, label=name)

axes[0].set_xlabel('Solde avant (émetteur)')
axes[0].set_ylabel('Solde après (émetteur)')
axes[0].set_title('Soldes émetteur : normal vs fraude')
axes[0].legend()

# Corrélation heatmap
num_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud']
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1])
axes[1].set_title('Matrice de corrélation')

plt.tight_layout()
plt.savefig('../reports/figures/ex1_suspicious_behavior.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Prétraitement & Feature Engineering

In [ ]:
# Feature engineering
df_feat = engineer_fraud_features(df)
df_feat = encode_transaction_type(df_feat)

# Supprimer les colonnes non pertinentes (identifiants)
drop_cols = ['nameOrig', 'nameDest', 'isFlaggedFraud']
df_feat = df_feat.drop(columns=drop_cols, errors='ignore')

print("Features disponibles :")
print(df_feat.columns.tolist())
print(f"\nShape après feature engineering : {df_feat.shape}")

In [ ]:
# Séparation features / cible
X = df_feat.drop(columns=['isFraud'])
y = df_feat['isFraud']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train : {X_train.shape} | Fraudes train : {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Test  : {X_test.shape}  | Fraudes test  : {y_test.sum()} ({y_test.mean()*100:.2f}%)")

## 3. Gestion du déséquilibre des classes

Les fraudes représentent ~0.1% des transactions. Stratégies testées :
- `class_weight='balanced'` dans les modèles
- SMOTE (oversampling de la classe minoritaire)
- Seuil de décision ajusté

In [ ]:
# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE sur train uniquement
smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Avant SMOTE  : {y_train.value_counts().to_dict()}")
print(f"Après SMOTE  : {pd.Series(y_train_resampled).value_counts().to_dict()}")

## 4. Modélisation

In [ ]:
# Comparaison par validation croisée
print("=== Validation croisée (ROC-AUC, 5 folds) ===")
cv_results = cross_validate_models(FRAUD_MODELS, X_train_scaled, y_train)

In [ ]:
# Entraînement et évaluation sur le test set
results = {}

for name, model in FRAUD_MODELS.items():
    y_pred, y_proba = train_and_evaluate(model, X_train_resampled, y_train_resampled,
                                          X_test_scaled, y_test, model_name=name)
    results[name] = evaluate_classifier(y_test, y_pred, y_proba, model_name=name)

In [ ]:
# Visualisation des courbes ROC
fig, ax = plt.subplots(figsize=(10, 7))

for name, model in FRAUD_MODELS.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        plot_roc_curve(y_test, y_proba, model_name=name, ax=ax)

ax.set_title('Courbes ROC — Comparaison des modèles')
plt.tight_layout()
plt.savefig('../reports/figures/ex1_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Réseau de neurones
import tensorflow as tf
from tensorflow import keras

nn_model = keras.Sequential([
    keras.layers.Input(shape=(X_train_resampled.shape[1],)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])

nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['AUC', 'Precision', 'Recall']
)

history = nn_model.fit(
    X_train_resampled, y_train_resampled,
    epochs=20, batch_size=512,
    validation_split=0.15,
    verbose=1
)

In [ ]:
# Courbe d'apprentissage du réseau de neurones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Perte (Loss)')
axes[0].legend()

axes[1].plot(history.history['auc'], label='Train')
axes[1].plot(history.history['val_auc'], label='Validation')
axes[1].set_title('AUC')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/ex1_nn_training.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Évaluation détaillée du meilleur modèle

In [ ]:
# Identifier le meilleur modèle selon ROC-AUC
best_model_name = max(results, key=lambda k: results[k]['roc_auc'] or 0)
best_model = FRAUD_MODELS[best_model_name]
print(f"Meilleur modèle : {best_model_name}")

y_pred_best = best_model.predict(X_test_scaled)
y_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_confusion_matrix(y_test, y_pred_best, ax=axes[0])
plot_roc_curve(y_test, y_proba_best, model_name=best_model_name, ax=axes[1])

plt.tight_layout()
plt.savefig('../reports/figures/ex1_best_model_eval.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Interprétabilité — SHAP

In [ ]:
import shap

# SHAP sur le meilleur modèle (TreeExplainer pour RF/XGB/LGBM)
explainer = shap.TreeExplainer(best_model)

# Échantillon pour performance
X_sample = pd.DataFrame(X_test_scaled[:2000], columns=X.columns)
shap_values = explainer.shap_values(X_sample)

# Pour les classifieurs binaires : shap_values peut être une liste [class0, class1]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_sample, plot_type='bar', show=False)
plt.title('Importance des features (SHAP — valeur absolue moyenne)')
plt.tight_layout()
plt.savefig('../reports/figures/ex1_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm : distribution des impacts
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_sample, show=False)
plt.title('Impact SHAP par feature et par observation')
plt.tight_layout()
plt.savefig('../reports/figures/ex1_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Explication d'une transaction frauduleuse spécifique
fraud_idx = y_test[y_test == 1].index[0]
fraud_pos = X_test.index.get_loc(fraud_idx)

shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    sv[fraud_pos],
    X_sample.iloc[fraud_pos],
    matplotlib=True,
)
plt.title('Explication SHAP — Transaction frauduleuse')
plt.tight_layout()
plt.savefig('../reports/figures/ex1_shap_force_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Analyse des faux positifs / faux négatifs

In [ ]:
X_test_df = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)
results_df = X_test_df.copy()
results_df['y_true'] = y_test.values
results_df['y_pred'] = y_pred_best
results_df['y_proba'] = y_proba_best

faux_positifs = results_df[(results_df['y_true'] == 0) & (results_df['y_pred'] == 1)]
faux_negatifs = results_df[(results_df['y_true'] == 1) & (results_df['y_pred'] == 0)]

print(f"Faux positifs : {len(faux_positifs)} (transactions normales classées frauduleuses)")
print(f"Faux négatifs : {len(faux_negatifs)} (fraudes non détectées)")

print("\n--- Caractéristiques des faux négatifs ---")
print(faux_negatifs.describe())

In [ ]:
# Distribution des probabilités prédites
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(results_df[results_df['y_true'] == 0]['y_proba'], bins=50,
             color='steelblue', alpha=0.7, label='Normal', density=True)
axes[0].hist(results_df[results_df['y_true'] == 1]['y_proba'], bins=50,
             color='tomato', alpha=0.7, label='Fraude', density=True)
axes[0].axvline(0.5, color='black', linestyle='--', label='Seuil 0.5')
axes[0].set_title('Distribution des probabilités prédites')
axes[0].set_xlabel('P(Fraude)')
axes[0].legend()

# Analyse du seuil optimal (F1)
from sklearn.metrics import f1_score
thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores = [f1_score(y_test, (y_proba_best >= t).astype(int)) for t in thresholds]
axes[1].plot(thresholds, f1_scores, 'o-', color='steelblue')
best_threshold = thresholds[np.argmax(f1_scores)]
axes[1].axvline(best_threshold, color='tomato', linestyle='--', label=f'Seuil optimal = {best_threshold:.2f}')
axes[1].set_title('F1-score en fonction du seuil de décision')
axes[1].set_xlabel('Seuil')
axes[1].set_ylabel('F1-score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/ex1_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSeuil optimal (F1 max) : {best_threshold:.2f} — F1 = {max(f1_scores):.4f}")

## Synthèse Exercice 1

**Points clés :**
- Le dataset est fortement déséquilibré (~0.1% de fraudes) → nécessite SMOTE + `class_weight`
- Les features `error_balance_orig` et `orig_zeroed` sont très discriminantes
- Le meilleur modèle selon ROC-AUC est : **[à compléter après exécution]**
- L'analyse SHAP révèle que les variables de solde et le type de transaction sont les plus importantes
- Le seuil de décision optimal (F1) est supérieur à 0.5 pour limiter les faux négatifs

**Recommandations business :**
- Surveiller en priorité les CASH_OUT et TRANSFER de montants élevés
- Mettre en place une revue manuelle des transactions à probabilité > seuil optimal
- Réentraîner le modèle régulièrement (dérive des comportements frauduleux)